# SSEBop AET — Aggregated to HRU Fabric

Visualize SSEBop actual evapotranspiration aggregated to HRU polygons
via gdptools from the USGS NHGF STAC catalog.

- **Variable:** `et` (Actual Evapotranspiration)
- **Units:** mm/month
- **Source:** SSEBop (doi:10.5066/P9L2YMV)
- **Resolution:** area-weighted mean per HRU polygon

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
NC_PATH = PROJECT / "data" / "aggregated" / "ssebop_agg_aet.nc"
FABRIC_PATH = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2_param_v2/gfv2/fabric/gfv2_nhru_merged.gpkg")
ID_COL = "nat_hru_id"


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"HRUs:       {ds.sizes[ID_COL]}")
print(f"Variables:  {list(ds.data_vars)}")
print(f"\net stats:")
print(f"  min:  {float(ds['et'].min()):.2f} mm/month")
print(f"  max:  {float(ds['et'].max()):.2f} mm/month")
print(f"  mean: {float(ds['et'].mean()):.2f} mm/month")
print(f"  NaN%: {float(ds['et'].isnull().mean()) * 100:.1f}%")


In [ ]:
# Load fabric geometry
fabric = gpd.read_parquet(FABRIC_PATH)
print(f"Fabric: {len(fabric)} HRUs, CRS={fabric.crs}")
print(f"Columns: {list(fabric.columns[:10])}...")


## July 2005 — AET map

In [ ]:
# Select a single month and merge with fabric geometry
month = "2005-07-01"
et_month = ds["et"].sel(time=month, method="nearest").to_dataframe().reset_index()

merged = fabric.merge(et_month[[ID_COL, "et"]], on=ID_COL, how="inner")
print(f"Merged: {len(merged)} HRUs with AET data")

fig, ax = plt.subplots(figsize=(16, 10))
merged.plot(
    column="et",
    ax=ax,
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"label": "AET (mm/month)", "shrink": 0.6},
    missing_kwds={"color": "lightgrey", "label": "No data"},
    vmin=0,
)
ax.set_title(f"SSEBop AET — July 2005", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"\nJuly 2005 AET stats:")
print(f"  min:  {merged['et'].min():.2f} mm/month")
print(f"  max:  {merged['et'].max():.2f} mm/month")
print(f"  mean: {merged['et'].mean():.2f} mm/month")


## Annual mean AET (2005)

In [ ]:
annual_mean = ds["et"].mean(dim="time").to_dataframe().reset_index()
merged_annual = fabric.merge(annual_mean[[ID_COL, "et"]], on=ID_COL, how="inner")

fig, ax = plt.subplots(figsize=(16, 10))
merged_annual.plot(
    column="et",
    ax=ax,
    cmap="YlOrRd",
    legend=True,
    legend_kwds={"label": "Mean AET (mm/month)", "shrink": 0.6},
    missing_kwds={"color": "lightgrey"},
    vmin=0,
)
ax.set_title("SSEBop Mean AET — 2005 Annual Mean", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


## Monthly time series (CONUS spatial mean)

In [ ]:
monthly_mean = ds["et"].mean(dim=ID_COL)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_mean.time.values, monthly_mean.values, marker="o", color="#c0392b")
ax.fill_between(monthly_mean.time.values, 0, monthly_mean.values, alpha=0.15, color="#c0392b")
ax.set_ylabel("Mean AET (mm/month)")
ax.set_title("CONUS-mean monthly SSEBop AET — 2005")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak month: {str(monthly_mean.time.values[monthly_mean.argmax().item()])[:7]}")
print(f"Peak AET:   {float(monthly_mean.max()):.2f} mm/month")


## Distribution of annual-mean AET across HRUs

In [ ]:
annual_vals = ds["et"].mean(dim="time").values
annual_vals = annual_vals[~np.isnan(annual_vals)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(annual_vals, bins=100, color="#c0392b", alpha=0.7, edgecolor="white")
ax.axvline(np.mean(annual_vals), color="black", linestyle="--", label=f"Mean: {np.mean(annual_vals):.1f}")
ax.axvline(np.median(annual_vals), color="blue", linestyle="--", label=f"Median: {np.median(annual_vals):.1f}")
ax.set_xlabel("Mean AET (mm/month)")
ax.set_ylabel("Number of HRUs")
ax.set_title("Distribution of Annual-Mean SSEBop AET Across HRUs")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Percentiles:")
for p in [5, 25, 50, 75, 95]:
    print(f"  P{p:02d}: {np.percentile(annual_vals, p):.2f} mm/month")


## Manifest provenance

In [ ]:
import json

manifest_path = PROJECT / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    entry = manifest.get("sources", {}).get("ssebop", {})
    for k, v in entry.items():
        print(f"{k:20s}: {v}")
else:
    print(f"manifest.json not found at {manifest_path}")


In [ ]:
ds.close()
